## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Theme park visitors by age and gender

**Population Pyramid** · a worked example from the StatNex teaching guide

**The story.** A family theme park counted a year of visitors by age group and gender, in hundreds of visitors.

**The question.** Who actually visits the park, and which groups is it missing?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/population_pyramid.csv` (21 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Age group,Male,Female
Age 0-4,380.0,365.0
Age 5-9,620.0,600.0
Age 10-14,700.0,690.0
Age 15-19,585.0,640.0
Age 20-24,660.0,720.0
Age 25-29,615.0,680.0
Age 30-34,720.0,790.0
Age 35-39,805.0,860.0
Age 40-44,640.0,690.0
Age 45-49,402.0,430.0
Age 50-54,281.0,300.0
Age 55-59,222.0,240.0
Age 60-64,168.0,182.0
Age 65-69,121.0,135.0
Age 70-74,79.0,92.0
Age 75-79,48.0,60.0
Age 80-84,26.0,35.0
Age 85-89,11.0,16.0
Age 90-94,4.0,6.0
Age 95-99,1.0,2.0
Age 100 or over,0.0,0.0
""")

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# Two groups counted over the same ordered classes, one row per class from the
# bottom of the pyramid up. A blank or text cell is 0.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
left_name, right_name = "Male", "Female"
item_name = "Age group"
unit = "hundred visitors" or ""
classes = data[[left_name, right_name]].apply(pd.to_numeric, errors="coerce").fillna(0)
classes.index = data[item_name].astype(str)

## Data

In [ ]:
# Written from the top of the pyramid down, so the last class comes first.
table = classes.iloc[::-1]
print(table)

## Composition

In [ ]:
# Each class as a share of its own group's total (which makes groups of different size
# comparable), the difference between the groups' counts (not their shares), and
# the ratio of the counts (blank where the right-hand count is 0). A ratio above 1 means
# more on the left in that class. These describe the counts only: they are not tests.
totals = classes.sum()
composition = pd.DataFrame({
    "share (left)": table[left_name] / totals[left_name],
    "share (right)": table[right_name] / totals[right_name],
    "difference (left - right)": table[left_name] - table[right_name],
    "ratio (left / right)": table[left_name] / table[right_name].where(table[right_name] != 0),
})
print()
print(composition)
print(f"\nTotal: {left_name} {totals[left_name]:,.0f}, {right_name} {totals[right_name]:,.0f}")

## Pyramid

In [ ]:
# The first class of the input at the base, the two groups back to back.
Path("figures").mkdir(exist_ok=True)
positions = np.arange(len(classes))
fig, ax = plt.subplots(figsize=(8, 0.3 * len(classes) + 2))
# The class names sit in a gap between the two groups' bars.
gap = 0.25 * classes.to_numpy().max()
ax.barh(positions, -classes[left_name], left=-gap, label=left_name)
ax.barh(positions, classes[right_name], left=gap, label=right_name)
for position, name in zip(positions, classes.index):
    ax.text(0, position, name, ha="center", va="center", fontsize=7)
ax.set_yticks([])
ax.set_xlabel(unit)
ax.legend()
fig.savefig("figures/pyramid.png", dpi=120, bbox_inches="tight")

## What the result says

A population pyramid puts male visitors on the left and female visitors on the right, one bar per age group. The counts are in hundreds of visitors. The widest bars are at age 35-39, with 805 male and 860 female, about 11% of each side. The bars stay wide from school age to the early forties.

Above that age they shrink fast. Age 45-49 has 402 and 430, far fewer than the 640 and 690 of age 40-44, and age 65-69 has only 121 and 135. The youngest group, age 0-4, is also small, with 380 and 365.

So the park draws mainly children and adults under 45. Toddlers and people over 45 are scarce. The pyramid cannot say whether that is the park's fault or simply who lives nearby.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.